In [33]:
import os
import time
import pandas as pd
import numpy as np
from DjData import jq
from DjData.tools import convert_stockcode

base_dir = r"c:\Users\AA\Desktop\海南板块\数据获取"

os.environ["REFRESH_KEY"] = "IOHYaC9ABE9.hraQFZCYaKD.fe8b2"

In [34]:

members = jq.get_concept_stocks(concept_code="882011.TI", date='2025-12-26')
print(len(members),members[:10])

pd.DataFrame({"code": members}).to_csv(os.path.join(base_dir, "members_2025-12-26.csv"), index=False, encoding="utf-8-sig")


27 ['000505', '000566', '000567', '000571', '000572', '000657', '000735', '000793', '000796', '000886']


In [36]:
out_path = r"c:\Users\AA\Desktop\海南板块\数据获取\prices_2025_daily.csv"
write_header = not os.path.exists(out_path) or os.stat(out_path).st_size == 0

members_jq = [convert_stockcode(str(c).zfill(6), kind="jq") for c in members]

for code in members_jq:
    df = jq.get_price(code, start_date="2025-01-01", end_date="2025-12-25", frequency="1d",
                      fields=["open","high","low","close","volume","money"])
    if isinstance(df, pd.DataFrame) and not df.empty:
        d = df.copy()
        d = d.reset_index()
        if "time" in d.columns:
            d = d.rename(columns={"time": "date"})
        elif "datetime" in d.columns:
            d = d.rename(columns={"datetime": "date"})
        elif "dealDate" in d.columns:
            d = d.rename(columns={"dealDate": "date"})
        elif "index" in d.columns:
            d = d.rename(columns={"index": "date"})
        d["code"] = code
        cols = ["date","code","open","high","low","close","volume","money"]
        d = d[[c for c in cols if c in d.columns]]
        d.to_csv(out_path, index=False, mode="a", encoding="utf-8-sig", header=write_header)
        write_header = False
    time.sleep(0.7)

print("成分股价格获取完成")

成分股价格获取完成


In [37]:
out_path = r"c:\Users\AA\Desktop\海南板块\数据获取\money_flow_2025_daily.csv"
write_header = not os.path.exists(out_path) or os.stat(out_path).st_size == 0
for i in members:
    flow_out = jq.get_money_flow(i,start_date="2025-01-01", end_date="2025-12-25")
    flow_out.to_csv(out_path, index=False, mode="a", encoding="utf-8-sig",header=write_header)
    write_header = False
    time.sleep(0.7)
print("成分股资金流获取完成")

成分股资金流获取完成


In [38]:
p = pd.read_csv(r"c:\Users\AA\Desktop\海南板块\数据获取\prices_2025_daily.csv", dtype={"code": str})

p["date"] = pd.to_datetime(p["date"], errors="coerce").dt.date
p["date"] = pd.to_datetime(p["date"])

pivot = p.pivot_table(index="date", columns="code", values="close", aggfunc="last").sort_index()
pivot = pivot.replace(0, np.nan)

daily_ret = pivot.pct_change(fill_method=None)
cum_factor = (1 + daily_ret.mean(axis=1, skipna=True).fillna(0)).cumprod()

out = cum_factor.rename("avg_return")
out.index.name = "date"

out.reset_index().to_csv(
    r"c:\Users\AA\Desktop\海南板块\数据获取\index_2025.csv",
    index=False,
    encoding="utf-8-sig"
)